# Bengali → Hindi translation with Qwen2.5-7B-Instruct — full run on Colab

Runtime ▸ Change runtime type ▸ **T4 GPU**, then Runtime ▸ Run all (≈ 1 h 15). Keep the tab open.

In [ ]:
%cd /content
!rm -rf bn-hi-llm-translation
!git clone -q https://github.com/AWARKEH1/bn-hi-llm-translation.git
%cd bn-hi-llm-translation
!pip install -q bitsandbytes peft sacrebleu
!nvidia-smi --query-gpu=name --format=csv,noheader

## 0. Unit tests (no GPU needed)

In [ ]:
!pip install -q pytest && python -m pytest -q

## 1. Zero-shot translation of the 509 test sentences

In [ ]:
!python -m bnhi.translate --mode zero_shot

## 2. Five-shot prompting (examples taken from the TRAIN split)

In [ ]:
!python -m bnhi.translate --mode few_shot --shots 5

## 3. QLoRA fine-tuning on the 503 TRAIN sentences

In [ ]:
!python -m bnhi.finetune --epochs 2

## 4. Fine-tuned model on the same test sentences

In [ ]:
!python -m bnhi.translate --mode finetuned

## 5. Logit lens: is there an English stage in the middle layers?

In [ ]:
!python -m bnhi.logit_lens

## 6. Results

In [ ]:
import json
from IPython.display import Image, display
print(json.dumps(json.load(open('results/metrics.json')), indent=2))
for f in ['finetune_loss.png', 'logit_lens.png']:
    display(Image(f'results/figures/{f}'))

In [ ]:
rd = lambda p: open(p, encoding='utf-8').read().splitlines()
src, ref = rd('data/test.bn'), rd('data/test.hi')
zs, ft = rd('results/translations_zero_shot.txt'), rd('results/translations_finetuned.txt')
for i in range(3):
    print(f'BN : {src[i]}\nREF: {ref[i]}\n0-shot: {zs[i]}\nfine-tuned: {ft[i]}\n')

## 7. Download the results (metrics, translations, figures)

In [ ]:
!zip -qr results.zip results -x 'results/lora_adapter/*'
from google.colab import files
files.download('results.zip')